# AND-01 · Coroutines under the hood

[Open in Colab](https://colab.research.google.com/github/BuiltBySwap/engineering-lab/blob/main/docs/topics/AND-01.ipynb)

**Status card** (edit it; the Progress and Tags pages are built from it):

```yaml
id: AND-01
title: "Coroutines under the hood"
track: android
date: 2026-10-06
status: done  # not-started | in-progress | partial | done
score: "4/5"
insight: "A coroutine is not a thread; it is a task with a bookmark (a continuation on the heap) that runs on a thread only while it is running, and a dispatcher picks the thread each time."
tags: [AND-01, android, kotlin, coroutines, threads]
pending:
  - "Write my own English sentences (bottleneck, workaround)"
  - "Mobile Lead rule in my own words"
  - "Fill the three ✏️ lines on this page"
```

## The problem
Threads are expensive: each reserves about 1 MB of stack, and most of the time they just wait. Callbacks free the thread but nest into callback hell.
✏️ One line in my own words: why did Kotlin add coroutines?

## The idea: a kitchen
- A **thread** is a cook. Cooks are few and expensive.
- A **coroutine** is an order card with a bookmark.
- **`suspend`** means: put the card down (bookmark kept) and take another card.
- A **dispatcher** is the head chef: it decides which cook picks up each card.

A coroutine runs on a thread **only while it is running**. While suspended it holds no thread.

## What the compiler does
`suspend` adds a hidden `Continuation` parameter and rewrites the function into a **state machine**: a `label` (the bookmark) plus saved local variables, kept in an object on the heap.

- A function with N suspension points has **N + 1** states.
- It returns either the real result or the marker `COROUTINE_SUSPENDED`; the caller compares the two, so one code path handles both.
- `suspend` never switches threads by itself. `withContext` and the dispatcher do.
- Saved locals get fields named by type: `L$0` (object), `I$0` (Int).

## What I measured (11-core Mac)
| Experiment | Result |
|---|---|
| 2 × `delay(1000)` on one thread | 1013 ms |
| 2 × `Thread.sleep(1000)` on one thread | 2013 ms |
| 100,000 coroutines each waiting 1 s | 1202 ms |
| 3,000 threads each sleeping 1 s | 1140 ms |
| 22 blocking tasks on `Default` (11 threads) | 2011 ms |
| 22 blocking tasks on `IO` | 1011 ms |
| 11 CPU-bound loops on 11 cores | 1008 ms |
| 33 CPU-bound loops on 11 cores | 3003 ms |

**Rule:** time ≈ ceil(tasks ÷ threads) × time each task holds its thread. Waiting overlaps for free; blocking and computing do not.

### Run it yourself (Kotlin)

This downloads the coroutines library the first time (`@file:DependsOn`). Needs internet.

In [ ]:
# Setup: run once per Colab session. It adds the %%kotlin cell magic.
!curl -sL -o kotlin_magic.py https://raw.githubusercontent.com/BuiltBySwap/engineering-lab/main/tools/kotlin_magic.py
%run kotlin_magic.py

In [ ]:
%%kotlin
@file:DependsOn("org.jetbrains.kotlinx:kotlinx-coroutines-core-jvm:1.10.2")

import kotlinx.coroutines.*
import kotlin.system.measureTimeMillis

runBlocking {
    val withDelay = measureTimeMillis {
        listOf(launch { delay(1000) }, launch { delay(1000) }).joinAll()
    }
    println("two delay(1000)        : $withDelay ms   (the waits overlap)")

    val withSleep = measureTimeMillis {
        listOf(launch { Thread.sleep(1000) }, launch { Thread.sleep(1000) }).joinAll()
    }
    println("two Thread.sleep(1000) : $withSleep ms   (one thread: they queue)")
}

## Rules I will use
- Never block `Main` or `Default`. Use a suspend API, or wrap the blocking call in `withContext(Dispatchers.IO)`.
- A suspend function should be main-safe: it chooses its own dispatcher.
- ✏️ One more rule, in my own words.

## Bugs and fixes
| Error or symptom | Cause | Fix |
|---|---|---|
| `SourceSet with name 'main' not found` | I ran a JVM `main()` inside an Android app module | Run JVM labs in a plain Kotlin project (`ktor-sample`) |

## Quiz and mistakes
- Score: 4/5 after retries.
- I thought `suspend` means async, and that a suspended coroutine holds an IO thread.
- I counted cases as delays instead of delays + 1, and wrote `count` as `L$1` (it is `I$0`).
- A `suspend` function that calls a blocking function still blocks the caller's thread.

## Where I used it
✏️ A ViewModel or repository from my own work, in one line.